# Annotating French manifestos

Session 3. Run the cells one by one, top to bottom. You get 24 whole manifestos (1958-2024) and a codebook with 12 themes.

## 1. Load the data and the codebook

In [ ]:
%pip install -q openai pandas pydantic

import json
from concurrent.futures import ThreadPoolExecutor

import pandas as pd, requests
from openai import OpenAI
from pydantic import BaseModel, ConfigDict

BASE = "https://raw.githubusercontent.com/subtilhugo/uzh-llm-socsci-2026/main/"
MANIFESTOS = pd.read_csv(BASE + "manifestos_france_sample.csv")
CODEBOOK = requests.get(BASE + "manifestos_france_codebook.txt").text
print(len(MANIFESTOS), "manifestos,", len(CODEBOOK), "characters of codebook")


In [ ]:
MANIFESTOS[["manifesto_id", "year", "source", "pdf_url"]].head()

In [ ]:
print(MANIFESTOS.year[0], "\n"); print(MANIFESTOS.text[0][:1500])

Read the codebook before anything else. **Would you code this manifesto the same way?**

In [ ]:
print(CODEBOOK)

## 2. Connect to the API

In the left sidebar, click the key icon (Secrets), add `OPENAI_API_KEY`, paste your key as the value, and switch on "Notebook access". Never paste a key into a cell. If you have no key, work with a neighbour.

In [ ]:
from google.colab import userdata

MODEL = "gpt-6-luna"
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"), max_retries=5)

THEMES = ["local_offer", "national_offer", "redistribution", "social_protection", "employment", "industry", "deindustrialisation", "public_services", "public_service_withdrawal", "territorial_inequality", "class_divisive", "class_unifying"]


class Code(BaseModel):
    model_config = ConfigDict(extra="forbid")
    present: bool
    quote: str


class Annotation(BaseModel):
    model_config = ConfigDict(extra="forbid")
    local_offer: Code
    national_offer: Code
    redistribution: Code
    social_protection: Code
    employment: Code
    industry: Code
    deindustrialisation: Code
    public_services: Code
    public_service_withdrawal: Code
    territorial_inequality: Code
    class_divisive: Code
    class_unifying: Code




def annotate(text):
    r = client.responses.parse(model=MODEL, max_output_tokens=3000, instructions=CODEBOOK, input=text,
                               text_format=Annotation, reasoning={"effort": "none"})
    return r.output_parsed


## 3. One manifesto

In [ ]:
annotate(MANIFESTOS.text[0]).industry

## 4. All of them

The same call on every manifesto, eight at a time.

In [ ]:
with ThreadPoolExecutor(8) as pool:
    ANSWERS = list(pool.map(annotate, MANIFESTOS.text))

In [ ]:
THEMES = list(Annotation.model_fields)
LABELS = pd.DataFrame([{t: getattr(a, t).present for t in THEMES} for a in ANSWERS])
LABELS.head()

In [ ]:
LABELS.mean().round(2).sort_values().plot.barh(title="Share of manifestos with each theme");

## 5. Now check it

Pick one manifesto and one theme. Read the quote, then open the original PDF. **Do you agree?**

In [ ]:
[(t, getattr(ANSWERS[0], t).quote) for t in THEMES if getattr(ANSWERS[0], t).present]

In [ ]:
MANIFESTOS.pdf_url[0]